# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** NIST SP 800-63B-4, *Digital Identity Guidelines: Authentication and Authenticator Management* (July 2025, 129 pages).
It is a regulatory text full of "SHALL", "SHALL NOT" and "except when" style rules, which makes it a good stress test for retrieval.

Both pipelines run over the **same chunks**:

| | Pipeline 1 (Baseline) | Pipeline 2 (Re-Ranking) |
|---|---|---|
| Candidate search | bi-encoder `all-MiniLM-L6-v2` + FAISS, top 10 by cosine | same top 10 |
| Ordering | cosine score | cross-encoder `ms-marco-MiniLM-L-6-v2` score |
| Sent to LLM | top 3 | top 3 after re-ranking |
| Prompt | `build_prompt()` | the same `build_prompt()` |

The LLM is Gemini (`gemini-3.5-flash`, temperature 0). The API key is read from the `GEMINI_API_KEY` environment variable.

In [1]:
# Transformers tries to import TensorFlow if it is installed; we only need PyTorch.
import os
os.environ["USE_TF"] = "0"

import bisect, collections, hashlib, json, statistics, textwrap, time
import faiss
import numpy as np
import pandas as pd
import pymupdf
import torch
from google import genai
from google.genai import types
from sentence_transformers import CrossEncoder, SentenceTransformer

PDF_PATH = "NIST.SP.800-63B-4.pdf"
BI_ENCODER = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL = "gemini-3.5-flash"

TOP_K = 10            # candidates pulled from the vector index
TOP_N = 3             # chunks that actually go into the prompt
CHUNK_TOKENS = 200    # chunk size, measured in bi-encoder tokens
OVERLAP_TOKENS = 40   # shared tokens between neighbouring chunks

pd.set_option("display.max_colwidth", 90)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Device: cpu


## Step 1: Load and clean the PDF

* The first 10 pages are the cover, abstract and table of contents. The TOC lines (e.g. *"Rate Limiting (Throttling) .... 41"*) look like real content to a retriever, so I start at the **Preface**.
* Every page repeats the same header (*"NIST SP 800-63B-4 / July 2025 / Digital Identity Guidelines ..."*). Any line that appears on more than half of the pages is treated as a running header and dropped.
* All pages are joined into one string, and I remember where each page starts so every chunk can be cited by page number.

In [2]:
pages = [page.get_text() for page in pymupdf.open(PDF_PATH)]
first_page = next(i for i, text in enumerate(pages) if "\nPreface\n" in text)

# Lines that repeat on most pages are headers/footers, not content.
line_counts = collections.Counter(line.strip() for text in pages for line in set(text.splitlines()))
boilerplate = {line for line, n in line_counts.items() if n > len(pages) / 2}
print("Removed header lines:", boilerplate)

full_text, page_starts = "", []
for i in range(first_page, len(pages)):
    page_starts.append(len(full_text))
    lines = [l.strip() for l in pages[i].splitlines() if l.strip() and l.strip() not in boilerplate]
    # Bullet glyphs come out as odd characters; collapse everything to single spaces.
    text = " ".join(lines).replace("\ufffd", " ").replace("\u2022", " ")
    full_text += " ".join(text.split()) + " "

print(f"Kept pages {first_page + 1}-{len(pages)}, {len(full_text):,} characters")

Removed header lines: {'NIST SP 800-63B-4', 'Digital Identity Guidelines', 'Authentication and Authenticator Management', 'July 2025'}
Kept pages 11-129, 257,138 characters


## Step 2: Chunking

The bi-encoder silently **truncates anything beyond 256 word-pieces**. NIST text is token-heavy (section numbers, "SP 800-63A", URLs), so my first try with 180-*word* windows produced chunks of up to 520 tokens, and the end of those chunks would have been invisible to the bi-encoder (but not to the cross-encoder, which reads 512). That would have made the comparison unfair.

So I chunk directly on the bi-encoder's tokens: windows of 200 tokens with 40 tokens of overlap. The tokenizer's offset mapping lets me slice the *original* text, so chunks keep their casing and punctuation.

In [3]:
bi_encoder = SentenceTransformer(BI_ENCODER)
tok = bi_encoder.tokenizer

# Tokenise the whole document once. verbose=False hides the "longer than 256" warning,
# which does not apply here because the full text is never fed to the model.
offsets = tok(full_text, add_special_tokens=False, return_offsets_mapping=True, verbose=False)["offset_mapping"]
chunks = []
for start in range(0, len(offsets), CHUNK_TOKENS - OVERLAP_TOKENS):
    end = min(start + CHUNK_TOKENS, len(offsets))
    char_start, char_end = offsets[start][0], offsets[end - 1][1]
    page = first_page + bisect.bisect_right(page_starts, char_start)  # 1-based PDF page
    chunks.append({"id": len(chunks), "page": page, "text": full_text[char_start:char_end]})
    if end == len(offsets):
        break

token_lengths = [len(tok(c["text"])["input_ids"]) for c in chunks]
# Check: no chunk may be cut off by the bi-encoder.
assert max(token_lengths) <= bi_encoder.max_seq_length, max(token_lengths)
print(f"{len(chunks)} chunks, longest = {max(token_lengths)} tokens (limit {bi_encoder.max_seq_length})")
print(f"\nExample: chunk 60 (page {chunks[60]['page']})\n" + textwrap.fill(chunks[60]["text"], 110))

337 chunks, longest = 203 tokens (limit 256)

Example: chunk 60 (page 32)
a wired connection to the PSTN that the verifier can call and dictate the out- of-band secret. For the
purposes of this definition, “wired connection” includes services such as cable providers that offer PSTN
services through other wired media and fiber via analog telephone adapters. For single-factor out-of-band
authenticators, if a secret is sent by the verifier to the out-of-band device, the device SHOULD NOT display
the authentication secret while it is locked by the owner. Rather, the device SHOULD require the presentation
and verification of a PIN, passcode, or biometric characteristic to view the secret. However, authenticators
SHOULD indicate the receipt of an authentication secret on a locked device. If the out-of-band authenticator
requests approval over the secondary communication channel rather than by presenting a secret that the
claimant transfers to the primary communication channel, it SHALL accep

## Step 3: Bi-encoder index (shared by both pipelines)

Each chunk is embedded once. Vectors are L2-normalised, so the inner product in `IndexFlatIP` **is** cosine similarity. A flat (exact) index is fine at this size; the tutorial notes exact search works up to roughly a million vectors.

In [4]:
chunk_vectors = bi_encoder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                                  batch_size=32, show_progress_bar=False)
index = faiss.IndexFlatIP(chunk_vectors.shape[1])
index.add(chunk_vectors)

def retrieve(query, k=TOP_K):
    """Pipeline 1 search: returns [(chunk_id, cosine)] best first."""
    q = bi_encoder.encode([query], normalize_embeddings=True)
    scores, ids = index.search(q, k)
    return list(zip(ids[0].tolist(), scores[0].tolist()))

hits = retrieve("What is the minimum password length?")
assert len(hits) == TOP_K and all(a[1] >= b[1] for a, b in zip(hits, hits[1:]))
print(index.ntotal, "vectors indexed; sample hits:", [(i, round(s, 3)) for i, s in hits[:3]])

337 vectors indexed; sample hits: [(266, 0.693), (268, 0.621), (218, 0.601)]


## Step 4: Cross-encoder, the shared prompt, and the LLM

* `rerank()` scores all 10 `(query, chunk)` pairs **jointly** with the cross-encoder and re-sorts them.
* `build_prompt()` follows the pattern from the RAG tutorial: numbered passages, cite them as [n], and say "I don't know" if the passages do not contain the answer. **Both answers go through this exact function**; only the list of chunk ids differs.
* `generate()` calls Gemini at temperature 0, retries on per-minute rate limits, and caches every answer.

In [5]:
cross_encoder = CrossEncoder(CROSS_ENCODER)
llm = genai.Client()  # picks up GEMINI_API_KEY from the environment

def rerank(query, candidates):
    """Pipeline 2: re-score the bi-encoder candidates and sort by cross-encoder score."""
    ids = [cid for cid, _ in candidates]
    scores = cross_encoder.predict([(query, chunks[cid]["text"]) for cid in ids])
    return sorted(zip(ids, scores.tolist()), key=lambda x: x[1], reverse=True)

def build_prompt(query, chunk_ids):
    passages = "\n\n".join(f"[{n}] (page {chunks[cid]['page']}) {chunks[cid]['text']}"
                           for n, cid in enumerate(chunk_ids, 1))
    return ("Answer using only the numbered passages below. Cite them like [1]. "
            "If the answer is not in the passages, say that you don't know.\n\n"
            f"Context:\n{passages}\n\nQuestion: {query}")

# The free Gemini tier only allows a small number of requests per day, so every answer is
# saved in llm_cache.json together with how long the real API call took. Re-running the
# notebook reuses those answers (and their measured latency) instead of calling the API again.
CACHE_PATH = "llm_cache.json"
cache = json.load(open(CACHE_PATH, encoding="utf-8")) if os.path.exists(CACHE_PATH) else {}

def generate(prompt, retries=5):
    """Returns (answer, seconds the API call took)."""
    key = hashlib.sha256(f"{LLM_MODEL}\n{prompt}".encode()).hexdigest()
    if key not in cache:
        for attempt in range(retries):
            try:
                t0 = time.perf_counter()
                reply = llm.models.generate_content(model=LLM_MODEL, contents=prompt,
                                                    config=types.GenerateContentConfig(temperature=0))
                cache[key] = {"answer": reply.text.strip(), "seconds": time.perf_counter() - t0}
                json.dump(cache, open(CACHE_PATH, "w", encoding="utf-8"), indent=1)
                break
            except Exception as e:
                # A per-minute limit clears quickly; a per-day limit does not, so stop.
                if attempt == retries - 1 or "PerDay" in str(e):
                    raise
                print(f"  LLM call failed ({str(e)[:50]}...), retrying in 30s")
                time.sleep(30)
    return cache[key]["answer"], cache[key]["seconds"]

In [6]:
def run_both(query):
    """Runs Pipeline 1 and Pipeline 2 for one query and times each stage."""
    t0 = time.perf_counter()
    baseline = retrieve(query)                       # P1: embed query + top-10 search
    t1 = time.perf_counter()
    reranked = rerank(query, baseline)               # P2: cross-encoder on those 10
    t2 = time.perf_counter()

    # Sanity checks: the re-ranker only reorders the same 10 candidates.
    assert {c for c, _ in reranked} == {c for c, _ in baseline}
    assert all(a[1] >= b[1] for a, b in zip(reranked, reranked[1:]))

    top3_base = [c for c, _ in baseline[:TOP_N]]
    top3_rerank = [c for c, _ in reranked[:TOP_N]]
    answer1, llm1 = generate(build_prompt(query, top3_base))
    answer2, llm2 = generate(build_prompt(query, top3_rerank))
    return {"query": query, "baseline": baseline, "reranked": reranked,
            "answer1": answer1, "answer2": answer2,
            "timings": {"retrieve": t1 - t0, "rerank": t2 - t1, "llm1": llm1, "llm2": llm2}}

demo = run_both("What is the minimum password length?")
print("Baseline top 3:", [c for c, _ in demo["baseline"][:3]])
print("Re-ranked top 3:", [c for c, _ in demo["reranked"][:3]])
print("\nAnswer 1:", demo["answer1"])
print("\nAnswer 2:", demo["answer2"])

Baseline top 3: [266, 268, 218]
Re-ranked top 3: [39, 266, 270]

Answer 1: Based on the provided passages, a specific numerical minimum password length is not provided; rather, the required minimum password length "depends on the threat model being addressed" [1].

Answer 2: Based on the provided passages, the minimum password length depends on the authentication process and threat model:

* **Single-factor authentication:** Passwords must be a minimum of 15 characters in length [1].
* **Multi-factor authentication:** Passwords may be shorter, but must be a minimum of eight characters in length [1].

Additionally, the minimum required password length depends on the threat model being addressed [2].


## Step 5: Failure-mode stress tests

Each query has one or more **gold phrases**: short pieces of the sentence(s) in the PDF that actually answer it. Any chunk containing a gold phrase counts as a gold chunk. Some rules are written twice (once in the normative section and once in the usability appendix), and with overlap one sentence can sit in two chunks.

I wrote the queries the way a normal user would ask, *not* in the document's wording. They cover the categories in the assignment:

* **Keyword trap**: the query shares words with many passages, but the answer uses different wording.
* **Negation / constraint**: the answer is a "SHALL NOT" rule.
* **Exception**: there is a general rule plus an "except/only if" condition.
* **Buried**: a short rule hidden in a long list of requirements.

I kept every query I tried, including the ones where re-ranking does not help, because those say something too.

In [7]:
QUERIES = [
    ("Keyword trap", "Can a website force me to include special characters and numbers in my password?",
     "mixtures of different character types"),
    ("Keyword trap", "Do users have to change their password every few months?",
     ("change passwords periodically", "changed arbitrarily (e.g., periodically) unless")),
    ("Negation", "Are password hints allowed?", "store a hint"),
    ("Negation", "Are security questions allowed when choosing a password?",
     "knowledge-based authentication (kba)"),
    ("Exception", "What is the minimum password length?", "minimum of 15 characters"),
    ("Exception", "Can SMS be used for out-of-band authentication?",
     "out-of-band verification is restricted"),
    ("Constraint", "Can a fingerprint alone be used to log in?", "biometrics shall only be used"),
    ("Buried", "Is pasting a password into the login field allowed?", "function when entering a password"),
    ("Buried", "Can a verifier check only part of the password?", "not truncate it"),
    ("Buried", "How many wrong guesses can an attacker make before the account is locked?",
     ("to no more than 100", "limit of 100 attempts")),
]

def gold_chunks(phrases):
    """Chunks that contain any of the gold phrases (a rule can be stated in two places)."""
    phrases = (phrases,) if isinstance(phrases, str) else phrases
    return {c["id"] for c in chunks if any(p.lower() in " ".join(c["text"].split()).lower() for p in phrases)}

for _, query, phrase in QUERIES:
    assert gold_chunks(phrase), f"gold phrase not found: {phrase}"
print("All gold phrases found in the corpus.")

All gold phrases found in the corpus.


In [8]:
results = []
for category, query, phrase in QUERIES:
    r = run_both(query)
    r.update(category=category, gold=gold_chunks(phrase))
    results.append(r)
print(len(results), "queries run through both pipelines")

10 queries run through both pipelines


In [9]:
def rank_table(r):
    """Top-10 candidates with their rank under each pipeline. * marks a gold chunk."""
    new_rank = {cid: n for n, (cid, _) in enumerate(r["reranked"], 1)}
    ce_score = dict(r["reranked"])
    rows = [{"chunk": cid, "page": chunks[cid]["page"], "gold": "*" if cid in r["gold"] else "",
             "baseline rank": n, "cosine": round(cos, 3),
             "rerank rank": new_rank[cid], "CE score": round(ce_score[cid], 2),
             "text": chunks[cid]["text"][:90] + "..."}
            for n, (cid, cos) in enumerate(r["baseline"], 1)]
    return pd.DataFrame(rows).set_index("chunk")

def show(r):
    print(f"[{r['category']}]  {r['query']}")
    display(rank_table(r))
    print("ANSWER 1 (baseline top 3:", [c for c, _ in r["baseline"][:3]], ")")
    print(textwrap.fill(r["answer1"], 110))
    print("\nANSWER 2 (re-ranked top 3:", [c for c, _ in r["reranked"][:3]], ")")
    print(textwrap.fill(r["answer2"], 110))
    print("=" * 110)

for r in results:
    show(r)

[Keyword trap]  Can a website force me to include special characters and numbers in my password?


,page,gold,baseline rank,cosine,rerank rank,CE score,text
chunk,,,,,,,
269,99,,1,0.576,1,3.85,likely to choose “Password1” if required to include an uppercase letter and a number o...
264,97,,2,0.567,4,-3.52,86 Appendix A. Strength of Passwords This appendix is informative. This appendix uses ...
263,97,,3,0.535,10,-8.66,system/files/ usability_and_biometrics_final2.pdf [UAX15] Whistler K (2022) Unicode No...
219,83,*,4,0.525,8,-4.87,"other composition rules (e.g., mixtures of different character types) on passwords. 72..."
218,83,*,5,0.507,9,-5.11,used password. Usability considerations for typical usage with a password manager incl...
41,25,,6,0.500,7,-4.71,"NOT prompt subscribers to use knowledge-based authentication (KBA) (e.g., “What was th..."
40,25,*,7,0.496,2,-0.94,[ISO/ISC 10646] characters in passwords. Each Unicode code point SHALL be counted as a...
39,25,,8,0.464,5,-4.02,of Passwords. 3.1.1.2. Password Verifiers The following requirements apply to password...
42,26,,9,0.452,3,-3.12,"advised that some endpoints may represent some characters differently, which would aff..."


ANSWER 1 (baseline top 3: [269, 264, 263] )
Yes. Online services use "composition rules" that require users to construct passwords using a mix of
character types, such as at least one digit (number), uppercase letter, and symbol [1], [2].

ANSWER 2 (re-ranked top 3: [269, 40, 42] )
Based on the provided passages, guidelines state that verifiers and Credential Service Providers (CSPs)
**shall not** impose composition rules requiring mixtures of different character types [2].   However, passage
[1] notes that in practice, some online services do require users to include uppercase letters, numbers, or
symbols [1].
[Keyword trap]  Do users have to change their password every few months?


,page,gold,baseline rank,cosine,rerank rank,CE score,text
chunk,,,,,,,
217,83,,1,0.480,4,-4.75,"sometimes referred to as a passphrase or PIN). Alternatively, they may use a password ..."
218,83,,2,0.443,2,-1.07,used password. Usability considerations for typical usage with a password manager incl...
264,97,,3,0.428,5,-7.46,86 Appendix A. Strength of Passwords This appendix is informative. This appendix uses ...
268,99,,4,0.412,3,-1.77,than those expected to resist only online attacks. Users should be encouraged to make ...
219,83,*,5,0.403,1,-0.18,"other composition rules (e.g., mixtures of different character types) on passwords. 72..."
317,121,,6,0.389,8,-8.97,is generally longer for added security. password A type of authenticator consisting of...
265,98,,7,0.383,6,-7.49,", and symbol). However, analyses of breached password databases reveal that the benefi..."
168,64,,8,0.369,10,-10.83,"., retained across a restart of the associated application or a reboot of the host dev..."
44,26,,9,0.369,7,-7.86,"be made on the subscriber account, as described in Sec. 3.2.2. Verifiers SHALL allow t..."


ANSWER 1 (baseline top 3: [217, 218, 264] )
Based on the provided text, no. The guidelines state: "Do not require that passwords be changed arbitrarily"
[2].

ANSWER 2 (re-ranked top 3: [219, 218, 268] )
No. Passwords should not be required to be changed arbitrarily or periodically unless there is a user request
or evidence of an authenticator compromise [1][2].
[Negation]  Are password hints allowed?


...

ANSWER 1 (baseline top 3: [264, 37, 218] )
Yes. Usability guidelines specify supporting copy and paste functionality in fields used for entering
passwords, including passphrases [3].

ANSWER 2 (re-ranked top 3: [44, 218, 37] )
Yes, pasting a password is allowed and recommended. Verifiers should permit claimants to use the "paste"
function when entering a password [1], and password fields should support copy and paste functionality [2].
[Buried]  Can a verifier check only part of the password?


,page,gold,baseline rank,cosine,rerank rank,CE score,text
chunk,,,,,,,
45,27,,1,0.665,1,3.14,their entry if they are in a location where their screen is unlikely to be observed. T...
44,26,,2,0.576,2,2.88,"be made on the subscriber account, as described in Sec. 3.2.2. Verifiers SHALL allow t..."
219,83,,3,0.565,9,-1.44,"other composition rules (e.g., mixtures of different character types) on passwords. 72..."
37,24,,4,0.550,6,-0.40,"s A password (sometimes referred to as a passphrase or, if numeric, a personal identif..."
52,28,,5,0.530,10,-2.13,as follows: Look-up secrets that are shorter than the minimum security strength specif...
272,100,,6,0.530,4,0.68,"authentication factor are often centrally verified by the CSP’s verifier, those that a..."
274,100,,7,0.524,8,-0.76,"dependent on the length and complexity of the password, these requirements are relaxed..."
38,24,,8,0.522,5,0.32,"In contrast to centrally verified passwords, activation secrets (similar to the 13 unl..."
40,25,,9,0.513,3,1.26,[ISO/ISC 10646] characters in passwords. Each Unicode code point SHALL be counted as a...


ANSWER 1 (baseline top 3: [45, 44, 219] )
I don't know. The provided passages do not contain information about whether a verifier can check only part of
a password.

ANSWER 2 (re-ranked top 3: [45, 44, 40] )
I don't know.
[Buried]  How many wrong guesses can an attacker make before the account is locked?


,page,gold,baseline rank,cosine,rerank rank,CE score,text
chunk,,,,,,,
267,98,,1,0.500,2,-0.34,to be complex enough that a reasonable number of attempts can be permitted with a low ...
94,41,*,2,0.470,1,0.70,"., single-factor cryptographic authenticator and centrally verified password), both au..."
266,98,,3,0.460,4,-5.26,"troke logging, phishing, and social engineering attacks are equally effective on lengt..."
274,100,,4,0.451,6,-8.97,"dependent on the length and complexity of the password, these requirements are relaxed..."
261,96,,5,0.448,10,-11.22,2023) Guidelines for Derived Personal Identity Verification (PIV) Credentials. (Nation...
185,70,,6,0.439,9,-10.71,attack to identify the correct password to decrypt the private key. Continued on next ...
186,71,,7,0.433,8,-9.60,to represent the bank. A password is revealed by the claimant at a bogus verifier webs...
316,121,,8,0.432,5,-8.60,an individual falsely denying having performed a particular transaction. offline attac...
181,69,,9,0.430,3,-5.15,": “Something you know” may be disclosed to an attacker. For example, the attacker may ..."


ANSWER 1 (baseline top 3: [267, 94, 266] )
Based on the provided passages, there is an upper bound limit of 100 attempts before authenticators are
disabled or a lockout occurs due to rate limiting, but agencies may impose lower limits [2].

ANSWER 2 (re-ranked top 3: [94, 267, 181] )
Based on the provided passages, the limit is an upper bound of 100 attempts, though agencies may impose lower
limits [1].


## Step 6: Rank-shift summary

Best rank of any gold chunk before and after re-ranking. **Rescued** means the gold chunk was outside the top 3 (ranks 4–10) for the baseline, so the LLM never saw it, and the cross-encoder pulled it into the top 3. "Not in top 10" means the bi-encoder missed it entirely, and a re-ranker cannot fix that, because it only reorders what it is given.

In [10]:
def best_rank(ranking, gold):
    ranks = [n for n, (cid, _) in enumerate(ranking, 1) if cid in gold]
    return min(ranks) if ranks else None

summary = []
for r in results:
    before, after = best_rank(r["baseline"], r["gold"]), best_rank(r["reranked"], r["gold"])
    if before is None:
        # Where did the bi-encoder actually put the answer in the whole corpus?
        full_rank = best_rank(retrieve(r["query"], k=len(chunks)), r["gold"])
        outcome = f"not in top 10 (recall miss, bi-encoder rank #{full_rank})"
    elif before <= TOP_N:
        outcome = "already in top 3" + ("" if after <= TOP_N else ", but pushed OUT by re-ranker")
    else:
        outcome = "RESCUED" if after <= TOP_N else "still outside top 3"
    moved = (f"chunk {[c for c, _ in r['reranked']][after - 1]}: #{before} -> #{after}"
             if before else "-")
    summary.append({"category": r["category"], "query": r["query"], "rank shift": moved, "outcome": outcome})

summary = pd.DataFrame(summary)
display(summary)
print("Rescued by the cross-encoder:", (summary["outcome"] == "RESCUED").sum(), "of", len(summary))

,category,query,rank shift,outcome
0,Keyword trap,Can a website force me to include special characters and numbers in my password?,chunk 40: #4 -> #2,RESCUED
1,Keyword trap,Do users have to change their password every few months?,chunk 219: #5 -> #1,RESCUED
2,Negation,Are password hints allowed?,chunk 40: #5 -> #1,RESCUED
3,Negation,Are security questions allowed when choosing a password?,-,"not in top 10 (recall miss, bi-encoder rank #11)"
4,Exception,What is the minimum password length?,chunk 39: #5 -> #1,RESCUED
5,Exception,Can SMS be used for out-of-band authentication?,-,"not in top 10 (recall miss, bi-encoder rank #14)"
6,Constraint,Can a fingerprint alone be used to log in?,chunk 99: #2 -> #2,already in top 3
7,Buried,Is pasting a password into the login field allowed?,chunk 44: #4 -> #1,RESCUED
8,Buried,Can a verifier check only part of the password?,-,"not in top 10 (recall miss, bi-encoder rank #18)"
9,Buried,How many wrong guesses can an attacker make before the account is locked?,chunk 94: #2 -> #1,already in top 3


Rescued by the cross-encoder: 5 of 10
